# Meeting 1: test

Tests `m1-real`, `m1-oversample` and the run named in `AUGMENT_RUN` on the test photos, then pushes the results to GitHub. Weights are restored from Google Drive, and the notebook stops if any are missing.

Run only after `AUGMENT_RUN` is set, and never change it afterwards.

In [3]:
import subprocess, sys
from getpass import getpass
from pathlib import Path
BRANCH = "excercises/1"  # the branch holding colab/ and runs/
REPO = Path("/content/dane_syntetyczne")
if not (REPO / ".git").exists():
    t = getpass("GitHub token (hidden): ")
    subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, f"https://{t}@github.com/tmsz/dane_syntetyczne.git", str(REPO)], check=True)
if subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", BRANCH]).returncode:
    raise RuntimeError(f"Branch {BRANCH!r} is not on GitHub. On the Mac: git push -u origin {BRANCH}")
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--quiet", "origin", BRANCH], check=True)
sys.path.insert(0, str(REPO / "colab"))
import colab_tools as ct
import importlib
importlib.reload(ct)  # re-running this cell picks up pulled changes


<module 'colab_tools' from '/content/dane_syntetyczne/colab/colab_tools.py'>

In [4]:
AUGMENT_RUN = "m1-augment-strong"  # augment() run kept after validation; never change after testing

assert AUGMENT_RUN in ("m1-augment", "m1-augment-strong"), f"bad AUGMENT_RUN: {AUGMENT_RUN!r}"
names = ["m1-real", "m1-oversample", AUGMENT_RUN]

ct.setup()
ct.mount_drive()
for name in names:
    ct.restore_models(name)  # model.pt <- Drive
    ct.test_run(name)
ct.check_devices(names)
ct.save_to_github(f"Meeting 1: test {', '.join(names)}")

Mounted at /content/drive
m1-real: 5 models restored from Drive

=== test m1-real ===
$ synth-test --run m1-real --device cuda
m1-real: 5 saved models on the test photos, cuda
  seed 0: Egyptian Mau recall 0.433, macro-F1 0.838
  seed 1: Egyptian Mau recall 0.500, macro-F1 0.866
  seed 2: Egyptian Mau recall 0.467, macro-F1 0.864
  seed 3: Egyptian Mau recall 0.433, macro-F1 0.847
  seed 4: Egyptian Mau recall 0.600, macro-F1 0.860
Test results of every seed are in /content/dane_syntetyczne/runs/m1-real/test-results.json
m1-oversample: 5 models restored from Drive

=== test m1-oversample ===
$ synth-test --run m1-oversample --device cuda
m1-oversample: 5 saved models on the test photos, cuda
  seed 0: Egyptian Mau recall 0.667, macro-F1 0.866
  seed 1: Egyptian Mau recall 0.733, macro-F1 0.895
  seed 2: Egyptian Mau recall 0.667, macro-F1 0.882
  seed 3: Egyptian Mau recall 0.700, macro-F1 0.891
  seed 4: Egyptian Mau recall 0.733, macro-F1 0.878
Test results of every seed are in /cont